# GraphRAG Pipeline — Financial Knowledge Graph
## FinBERT NER → SSHG → GCN → LLM Relation Extraction → Neo4j → Leiden → QA

**Pipeline overview (two parts):**

| Part | Steps |
|------|-------|
| **Part 1 – KG Creation (offline)** | Preprocessing → FinBERT+BIO NER → SSHG construction → GCN propagation → Temporal/Numeric anchoring → LLM relation extraction (Llama 3.2 3B) → Neo4j graph → Leiden communities |
| **Part 2 – Retrieval & QA (inference)** | Query type classification → Router (local/global/hybrid) → Mistral-NeMo answer generation → RAG Triad evaluation |

**Fairness vs baselines:** same 5 000-article Bloomberg corpus, same 20 questions from `qa_set.json`, same `llama3.2:3b` judge.

## 0. Environment Setup

Install all dependencies. Run once.

In [1]:
# Core NLP + graph
!pip install transformers torch spacy networkx sentence-transformers -q
!pip install python-dateutil requests datasets tqdm rapidfuzz -q
# Graph DB + community detection
!pip install neo4j leidenalg python-igraph -q
# spaCy language model for dependency parsing
!python -m spacy download en_core_web_sm -q
print("All dependencies installed.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
All dependencies installed.


## 1. Configuration & Helpers

All tuneable parameters in one place. Ollama connectivity is verified on import.

In [2]:
from __future__ import annotations
import json, os, re, time, logging, requests
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from datasets import load_dataset
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

CFG = {
    # Ollama
    "ollama_url"         : os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434"),
    "llm_model"          : "llama3.2:3b",      # NER judge + relation extraction + evaluation
    "answer_model"       : "mistral-nemo",      # answer generation
    # Models
    "plm_name"           : "ProsusAI/finbert",
    "embed_model"        : "sentence-transformers/all-MiniLM-L6-v2",
    # Corpus
    "n_articles"         : 5000,
    "chunk_size"         : 512,   # chars
    "chunk_overlap"      : 64,
    # GCN
    "gcn_hidden_dim"     : 256,
    "gcn_output_dim"     : 256,
    # Retrieval
    "semantic_threshold" : 0.7,
    "top_k_global"       : 5,
    "hop"                : 2,
    # Neo4j (optional)
    "neo4j_uri"          : os.environ.get("NEO4J_URI",      "bolt://localhost:7687"),
    "neo4j_user"         : os.environ.get("NEO4J_USER",     "neo4j"),
    "neo4j_password"     : os.environ.get("NEO4J_PASSWORD", "password"),
    # I/O
    "qa_set_path"        : "qa_set.json",
    "results_path"       : "graphrag_results.json",
    "temperature"        : 0.0,
    # KG build: set to len(article_chunks) for full run; lower for quick demo
    "process_n_chunks"   : 500,
    # Community summaries cap
    "max_communities"    : 30,
}

def ollama(model: str, prompt: str, timeout: int = 120) -> str:
    """Call Ollama /api/generate. Returns response text or '[ERROR: ...]'."""
    try:
        r = requests.post(
            f"{CFG['ollama_url']}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"temperature": CFG["temperature"]}},
            timeout=timeout,
        )
        r.raise_for_status()
        return r.json().get("response", "").strip()
    except Exception as e:
        logger.error("Ollama call failed: %s", e)
        return f"[ERROR: {e}]"

# Verify connectivity and resolve model tags
def _resolve_model(requested: str, available: list[str]) -> str:
    if requested in available: return requested
    base = requested.split(":")[0]
    for name in available:
        if name.split(":")[0] == base: return name
    return requested

try:
    _r = requests.get(f"{CFG['ollama_url']}/api/tags", timeout=5)
    _models = [m["name"] for m in _r.json().get("models", [])]
    print(f"Ollama up at {CFG['ollama_url']}. Models: {_models}")
    CFG["llm_model"]    = _resolve_model(CFG["llm_model"],    _models)
    CFG["answer_model"] = _resolve_model(CFG["answer_model"], _models)
    print(f"Using LLM={CFG['llm_model']}, answer={CFG['answer_model']}")
except Exception as e:
    print(f"WARNING: Ollama not reachable — {e}. Start with: ollama serve")

/home/aries/projects/academic/GenAIProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ollama up at http://127.0.0.1:11434. Models: ['mistral-nemo:latest', 'llama3.2:3b']
Using LLM=llama3.2:3b, answer=mistral-nemo:latest


## 2. Dataset Loading — Bloomberg Financial News 120K

Stream the first `n_articles` articles. Ticker symbols are detected via regex and stored for later anchoring.

In [3]:
TICKER_PAT = re.compile(r'\(([A-Z]{1,5}(?::[A-Z]{2})?)\)|\$([A-Z]{1,5})\b')

print(f"Streaming {CFG['n_articles']:,} articles from Bloomberg Financial News 120K...")
_stream = load_dataset("XJCEO/bloomberg_financial_news_120k", split="train", streaming=True)

articles: List[Dict] = []
for _i, _row in enumerate(tqdm(_stream, total=CFG["n_articles"], desc="Loading")):
    if _i >= CFG["n_articles"]:
        break
    _text = _row.get("Article") or _row.get("article") or ""
    if not isinstance(_text, str) or len(_text.strip()) < 30:
        continue
    _tickers = list({m.group(1) or m.group(2)
                     for m in TICKER_PAT.finditer(_text) if m.group(1) or m.group(2)})
    articles.append({
        "article_id" : _i,
        "headline"   : str(_row.get("Headline", "")),
        "text"       : re.sub(r'\s+', ' ', _text).strip(),
        "tickers"    : _tickers,
        "pub_date"   : str(_row.get("Date", "")) or None,
    })

print(f"Loaded {len(articles):,} articles")
print(f"Sample: [{articles[0]['article_id']}] {articles[0]['headline']} | tickers={articles[0]['tickers']}")

Streaming 5,000 articles from Bloomberg Financial News 120K...


2026-04-15 00:18:31,186 INFO HTTP Request: HEAD https://huggingface.co/datasets/XJCEO/bloomberg_financial_news_120k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-15 00:18:31,319 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/XJCEO/bloomberg_financial_news_120k/bc24a97ae247bb95475cbf1a141e341c4f5ad91b/README.md "HTTP/1.1 200 OK"
2026-04-15 00:18:31,465 INFO HTTP Request: HEAD https://huggingface.co/datasets/XJCEO/bloomberg_financial_news_120k/resolve/bc24a97ae247bb95475cbf1a141e341c4f5ad91b/bloomberg_financial_news_120k.py "HTTP/1.1 404 Not Found"
2026-04-15 00:18:31,904 INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/XJCEO/bloomberg_financial_news_120k/XJCEO/bloomberg_financial_news_120k.py "HTTP/1.1 404 Not Found"
2026-04-15 00:18:33,049 INFO HTTP Request: GET https://huggingface.co/api/datasets/XJCEO/bloomberg_financial_news_120k/revision/bc24a97ae247bb95475cbf1a141e341c4f5ad91b "HTTP/1.1 200 OK"

Loaded 5,000 articles
Sample: [0] Marriott Cuts Profit Forecast as Demand Weakens Overseas | tickers=['MAR', 'FBRC', 'HOT', 'H']


## 3. Preprocessing — Chunking

Split each article into overlapping fixed-size character windows.
- `chunk_size = 512` chars ≈ 2–3 sentences — enough for a full entity-relation triple
- `chunk_overlap = 64` chars — prevents relations being split at chunk boundaries

In [4]:
def chunk_text(text: str, size: int = CFG["chunk_size"],
               overlap: int = CFG["chunk_overlap"]) -> List[str]:
    """Fixed-size character chunking with overlap."""
    chunks, i = [], 0
    while i < len(text):
        c = text[i : i + size].strip()
        if len(c) > 20:
            chunks.append(c)
        i += max(1, size - overlap)
    return chunks

article_chunks: List[Dict] = []
for art in articles:
    for j, chunk in enumerate(chunk_text(art["text"])):
        article_chunks.append({
            "chunk_id"   : f"{art['article_id']}_{j}",
            "article_id" : art["article_id"],
            "headline"   : art["headline"],
            "text"       : chunk,
            "pub_date"   : art["pub_date"],
            "tickers"    : art["tickers"],
        })

print(f"Total chunks : {len(article_chunks):,}")
print(f"Avg / article: {len(article_chunks)/len(articles):.1f}")

Total chunks : 33,131
Avg / article: 6.6


## 4. Named Entity Extraction — FinBERT + BIO Head

`ProsusAI/finbert` is pre-trained on financial corpora (Reuters, Bloomberg, earnings calls).
A linear BIO classification head is placed on top of the last hidden state.
Token-level BIO predictions are decoded into entity spans; each span's embedding
is obtained by mean-pooling its constituent token embeddings from FinBERT.

> **Note:** The BIO head is randomly initialised here. For production accuracy,
> fine-tune it on a labelled financial NER dataset (e.g. FiNER-139).

In [5]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# Populated in the SSHG setup cell.
nlp_ner = None
nlp_dep = None

ENTITY_TYPES = ["ORG", "PER", "MONEY", "DATE", "GPE", "PRODUCT", "EVENT", "NORP"]
BIO_LABELS   = ["O"] + [f"B-{t}" for t in ENTITY_TYPES] + [f"I-{t}" for t in ENTITY_TYPES]
LABEL2ID     = {l: i for i, l in enumerate(BIO_LABELS)}
ID2LABEL     = {i: l for l, i in LABEL2ID.items()}

# spaCy label -> our entity type
_SPACY_TYPE_MAP = {
    "ORG":"ORG", "PERSON":"PER", "MONEY":"MONEY", "DATE":"DATE",
    "GPE":"GPE", "LOC":"GPE", "PRODUCT":"PRODUCT", "EVENT":"EVENT",
    "NORP":"NORP", "FAC":"ORG",
}
_KEEP_TYPES = set(ENTITY_TYPES)

# Strict blacklist to suppress obvious retrieval garbage as entities.
_ENTITY_STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "at", "by", "for", "to", "is", "are",
    "was", "were", "be", "been", "being", "have", "has", "had", "do", "does",
    "did", "will", "would", "could", "should", "may", "might", "shall", "what",
    "which", "who", "whom", "whose", "when", "where", "why", "how", "this",
    "that", "these", "those", "it", "its", "year", "quarter", "percent",
    "period", "time", "day", "month", "week", "date", "new", "first", "last",
    "between", "after", "before", "during", "over", "under", "into", "from",
    "planned", "outstanding", "list", "friend", "s",
}

def _clean_entity_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip(" \t\n\r\"'`()[]{}.,:;!?-")
    return text

def is_valid_entity_name(name: str, entity_type: Optional[str] = None) -> bool:
    n = _clean_entity_text(name)
    if len(n) < 3:
        return False
    if n.lower() in _ENTITY_STOPWORDS:
        return False
    if not re.search(r"[A-Za-z]", n):
        return False
    if re.fullmatch(r"[\W_]+", n):
        return False
    if entity_type not in {"MONEY", "DATE"} and n.lower() == n and len(n.split()) == 1:
        return False
    if entity_type == "MONEY" and not re.search(r"\d|\$|%|percent|bps|basis", n.lower()):
        return False
    if entity_type == "DATE" and not re.search(r"\d|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|q[1-4]", n.lower()):
        return False
    return True

def _safe_device() -> torch.device:
    """Select CUDA only if functional (CC < 7.0 falls back to CPU)."""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        _t = torch.zeros(2, 2, device="cuda")
        _ = (_t @ _t).item()
        cc = torch.cuda.get_device_capability()
        if cc[0] < 7 and not CFG.get("force_cuda", False):
            print(f"WARNING: CUDA CC {cc[0]}.{cc[1]} < 7.0 — using CPU.")
            return torch.device("cpu")
        return torch.device("cuda")
    except Exception as _e:
        print(f"CUDA test failed ({_e}) — CPU fallback.")
        return torch.device("cpu")

device = _safe_device()
print(f"Device: {device}")

print(f"Loading FinBERT: {CFG['plm_name']} ...")
tokenizer = AutoTokenizer.from_pretrained(CFG["plm_name"])
finbert   = AutoModel.from_pretrained(CFG["plm_name"])
finbert.eval()
finbert   = finbert.to(device)
print(f"FinBERT on {device}  |  hidden_size={finbert.config.hidden_size}")

# NOTE: BIO head omitted — randomly initialised weights produce garbage tags.
# NER spans are from spaCy pre-trained NER; FinBERT is used for embeddings.

@dataclass
class EntityMention:
    text        : str
    entity_type : str
    start       : int
    end         : int
    embedding   : Optional[np.ndarray] = field(default=None, repr=False)

@torch.no_grad()
def _finbert_token_embeddings(text: str, max_len: int = 512):
    """FinBERT token embeddings + offset mapping."""
    enc     = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        return_offsets_mapping=True,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc     = {k: v.to(device) for k, v in enc.items()}
    hidden  = finbert(**enc).last_hidden_state[0]
    return hidden.cpu().float().numpy(), offsets

def extract_entities_bio(text: str, max_len: int = 512):
    """
    Entity extraction:
      1. spaCy pre-trained NER -> entity spans
      2. FinBERT token embeddings -> mean-pool per span (for SSHG / GCN)
    Returns (List[EntityMention], token_embeddings np.ndarray).
    """
    if nlp_ner is None:
        raise RuntimeError("spaCy NER pipeline is not loaded. Run the SSHG setup cell first.")

    doc = nlp_ner(text[:1000])
    tok_np, offsets = _finbert_token_embeddings(text, max_len)

    def _pool(cs: int, ce: int) -> np.ndarray:
        idxs = [j for j, (a, b) in enumerate(offsets) if a < ce and b > cs and (b - a) > 0]
        return tok_np[idxs].mean(axis=0) if idxs else tok_np.mean(axis=0)

    entities = []
    seen = set()
    for ent in doc.ents:
        etype = _SPACY_TYPE_MAP.get(ent.label_)
        txt = _clean_entity_text(ent.text)
        if etype in _KEEP_TYPES and is_valid_entity_name(txt, etype):
            key = (txt.lower(), etype, ent.start_char, ent.end_char)
            if key in seen:
                continue
            seen.add(key)
            entities.append(
                EntityMention(
                    text=txt,
                    entity_type=etype,
                    start=ent.start_char,
                    end=ent.end_char,
                    embedding=_pool(ent.start_char, ent.end_char),
                )
            )
    return entities, tok_np

print("FinBERT encoder ready (NER via spaCy, embeddings via FinBERT).")
print(f"hidden_size={finbert.config.hidden_size}  |  entity_types={len(ENTITY_TYPES)}")

/home/aries/projects/academic/GenAIProject/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce GTX 1050 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/aries/projects/academic/GenAIProject/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce GTX 1050 with CUDA capability sm_61 is not compatible 

CUDA test failed (CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
) — CPU fallback.
Device: cpu
Loading FinBERT: ProsusAI/finbert ...


2026-04-15 00:18:54,155 INFO HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 00:18:54,208 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-04-15 00:18:54,358 INFO HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 00:18:54,411 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-15 00:18:54,559 INFO HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-15 00:18:54,703 INFO HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=

FinBERT on cpu  |  hidden_size=768
FinBERT encoder ready (NER via spaCy, embeddings via FinBERT).
hidden_size=768  |  entity_types=8


## 5. SSHG — Syntax-Semantics Hybrid Graph

The SSHG operates at the **entity-mention level** (not raw tokens).

| Edge type | How added |
|-----------|-----------|
| **Syntactic** | Two entity mentions whose head tokens are connected by a dependency arc (`nsubj`, `dobj`, `prep`, `appos`, …) |
| **Semantic — co-occurrence** | Two entity mentions appear in the same sentence |
| **Semantic — cosine similarity** | Cosine similarity of FinBERT pooled embeddings ≥ threshold (0.7) |

In [6]:
import networkx as nx
import spacy

print("Loading spaCy pipelines ...")
nlp_ner = spacy.load("en_core_web_sm")                    # with NER for entity spans
nlp_dep = spacy.load("en_core_web_sm", disable=["ner"])  # without NER for faster dependency parsing
print("spaCy pipelines loaded.")

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 1e-9 and nb > 1e-9 else 0.0

SYN_DEPS = {"nsubj", "dobj", "pobj", "prep", "appos", "compound", "nsubjpass", "csubj", "attr"}

def build_sshg(text: str, entities: List[EntityMention], threshold: float = CFG["semantic_threshold"]) -> nx.DiGraph:
    """Build Syntax-Semantics Hybrid Graph with entity-mention nodes."""
    G = nx.DiGraph()
    if not entities:
        return G

    for i, ent in enumerate(entities):
        G.add_node(i, text=ent.text, type=ent.entity_type, start=ent.start, end=ent.end)

    doc = nlp_dep(text[:1000])  # cap for speed

    def head_tok(ent: EntityMention) -> Optional[int]:
        """spaCy token index of the head token of an entity span."""
        cands = [t for t in doc if t.idx >= ent.start and t.idx < ent.end]
        return cands[0].head.i if cands else None

    heads = [head_tok(e) for e in entities]
    sents = [(s.start_char, s.end_char) for s in doc.sents]

    for i, (ei, hi) in enumerate(zip(entities, heads)):
        for j, (ej, hj) in enumerate(zip(entities, heads)):
            if i >= j:
                continue

            # Syntactic edge
            if hi is not None and hj is not None:
                ti, tj = doc[hi], doc[hj]
                if ti.head.i == hj and ti.dep_ in SYN_DEPS:
                    G.add_edge(i, j, edge_type="syntactic", dep=ti.dep_, weight=1.0)
                elif tj.head.i == hi and tj.dep_ in SYN_DEPS:
                    G.add_edge(j, i, edge_type="syntactic", dep=tj.dep_, weight=1.0)

            # Co-occurrence (same sentence)
            same = any(s[0] <= ei.start < s[1] and s[0] <= ej.start < s[1] for s in sents)
            if same and not G.has_edge(i, j):
                G.add_edge(i, j, edge_type="cooccurrence", weight=0.5)
                G.add_edge(j, i, edge_type="cooccurrence", weight=0.5)

            # Cosine similarity
            if ei.embedding is not None and ej.embedding is not None:
                sim = cosine_sim(ei.embedding, ej.embedding)
                if sim >= threshold and not G.has_edge(i, j):
                    G.add_edge(i, j, edge_type="semantic_sim", weight=float(sim))
                    G.add_edge(j, i, edge_type="semantic_sim", weight=float(sim))

    return G

# NER sanity check
_ents, _embs = extract_entities_bio(articles[0]["text"][:500])
print(f"\nNER check: {len(_ents)} entities (spaCy NER + FinBERT embeddings)")
for _e in _ents[:6]:
    print(f"  [{_e.entity_type:7s}] '{_e.text}'")

_sshg = build_sshg(articles[0]["text"][:500], _ents)
_etypes = set(nx.get_edge_attributes(_sshg, "edge_type").values())
print(f"\nSSHG: {_sshg.number_of_nodes()} nodes, {_sshg.number_of_edges()} edges | types={_etypes}")

Loading spaCy pipelines ...
spaCy pipelines loaded.

NER check: 9 entities (spaCy NER + FinBERT embeddings)
  [ORG    ] 'Marriott International Inc'
  [ORG    ] 'MAR'
  [GPE    ] 'U.S'
  [MONEY  ] '42 cents'
  [MONEY  ] '46 cents'
  [MONEY  ] '49-cent'

SSHG: 9 nodes, 20 edges | types={'cooccurrence'}


## 6. GCN Propagation — 2-Layer Graph Convolutional Network

$$h_i^{(l+1)} = \sigma\!\left(\hat{A}\, H^{(l)}\, W^{(l)}\right)$$

where $\hat{A} = D^{-1/2}(A+I)D^{-1/2}$ is the symmetrically normalised adjacency with self-loops.
Node features are initialised from FinBERT pooled embeddings.

In [7]:
class GCNLayer(nn.Module):
    """GCN layer: Â X W with LayerNorm + ReLU."""
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
        self.act  = nn.ReLU()
    def forward(self, X: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(A_hat @ self.W(X)))

class TwoLayerGCN(nn.Module):
    def __init__(self, in_dim: int, hid: int, out_dim: int):
        super().__init__()
        self.l1 = GCNLayer(in_dim, hid)
        self.l2 = GCNLayer(hid, out_dim)
    def forward(self, X: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        return self.l2(self.l1(X, A_hat), A_hat)

def norm_adj(G: nx.DiGraph, n: int) -> torch.Tensor:
    """Symmetric normalised adjacency Â = D^{-½}(A+I)D^{-½}."""
    A = np.zeros((n, n), dtype=np.float32)
    for u, v, d in G.edges(data=True):
        if u < n and v < n:
            A[u, v] = d.get("weight", 1.0)
    A = (A + A.T) / 2 + np.eye(n, dtype=np.float32)
    D_inv_sqrt = np.diag(A.sum(1).clip(1e-9) ** -0.5)
    return torch.tensor(D_inv_sqrt @ A @ D_inv_sqrt, dtype=torch.float32)

gcn = TwoLayerGCN(finbert.config.hidden_size,
                  CFG["gcn_hidden_dim"],
                  CFG["gcn_output_dim"]).to(device)
gcn.eval()
print(f"GCN: {finbert.config.hidden_size} -> {CFG['gcn_hidden_dim']} -> {CFG['gcn_output_dim']}")

@torch.no_grad()
def apply_gcn(entities: List[EntityMention], sshg: nx.DiGraph) -> np.ndarray:
    """Apply 2-layer GCN; returns refined embeddings (N, gcn_output_dim)."""
    n = len(entities)
    if n == 0:
        return np.zeros((0, CFG["gcn_output_dim"]), dtype=np.float32)
    H = CFG["gcn_hidden_dim"]   # use gcn_output_dim as feat_dim too
    feat_dim = finbert.config.hidden_size
    X = np.zeros((n, feat_dim), dtype=np.float32)
    for i, ent in enumerate(entities):
        if ent.embedding is not None:
            e = ent.embedding[:feat_dim]
            X[i, :len(e)] = e
    Xt    = torch.tensor(X, dtype=torch.float32).to(device)
    A_hat = norm_adj(sshg, n).to(device)
    return gcn(Xt, A_hat).cpu().numpy()

_refined = apply_gcn(_ents, _sshg)
print(f"GCN output: {_refined.shape}")

GCN: 768 -> 256 -> 256
GCN output: (9, 256)


## 7. Temporal Anchoring, Provenance & Numeric Facts

- **Dates** are extracted with regex + `dateutil` and normalised to ISO 8601.
  The document publication date is used as a fallback.
- **Provenance** attaches `chunk_id`, source sentence, `doc_id`, and `doc_timestamp` to every relation.
- **Numeric facts** (revenue, deal values, percentages) are stored as edge properties with
  `relation_type = "HAS_NUMERIC_FACT"` on self-loop edges.

In [8]:
from dateutil import parser as dateparser

_DATE_PATS = [
    re.compile(r'\b(\d{4}-\d{2}-\d{2})\b'),
    re.compile(r'\b(Q[1-4]\s+\d{4})\b'),
    re.compile(r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
               r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|'
               r'Dec(?:ember)?)\s+\d{1,2},?\s+\d{4}\b', re.IGNORECASE),
    re.compile(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}\b',
               re.IGNORECASE),
    re.compile(r'\b(\d{4})\b'),
]
_NUM_PAT = re.compile(
    r'(\$\s*[\d,]+(?:\.\d+)?(?:\s*(?:billion|million|thousand|[BMK]))?|'
    r'[\d,]+(?:\.\d+)?\s*(?:percent|%|basis\s+points?|bps?))',
    re.IGNORECASE,
)

def extract_dates(text: str, fallback: Optional[str] = None) -> List[str]:
    seen, out = set(), []
    for pat in _DATE_PATS:
        for m in pat.finditer(text):
            raw = m.group(0).strip()
            if raw in seen: continue
            seen.add(raw)
            try:
                parsed = dateparser.parse(raw)
                out.append(parsed.strftime("%Y-%m-%d") if parsed else raw)
            except Exception:
                out.append(raw)
    return out or ([fallback] if fallback else [])

@dataclass
class Provenance:
    chunk_id      : str
    sentence      : str
    doc_id        : int
    doc_timestamp : Optional[str]
    def to_dict(self) -> Dict:
        return {"chunk_id": self.chunk_id, "sentence": self.sentence,
                "doc_id": self.doc_id, "doc_timestamp": self.doc_timestamp}

@dataclass
class NumericFact:
    metric        : str
    raw_value     : str
    unit          : str
    time_interval : Optional[str]
    def to_dict(self) -> Dict:
        return {"metric": self.metric, "raw_value": self.raw_value,
                "unit": self.unit, "time_interval": self.time_interval}

_METRIC_WORDS = ["revenue","earnings","profit","sales","deal","acquisition",
                 "value","loss","income","gain","dividend","assets"]

def extract_numeric_facts(text: str, dates: List[str]) -> List[NumericFact]:
    facts = []
    for m in _NUM_PAT.finditer(text):
        raw  = m.group(0).strip()
        unit = ("USD"     if "$" in raw
                else "percent"  if "%" in raw or "percent" in raw.lower()
                else "bps"      if "basis" in raw.lower()
                else "number")
        ctx    = text[max(0, m.start()-40): m.start()].lower()
        metric = next((w for w in _METRIC_WORDS if w in ctx), "value")
        facts.append(NumericFact(metric, raw, unit, dates[0] if dates else None))
    return facts

# Test
_dates = extract_dates(articles[0]["text"][:500], articles[0]["pub_date"])
_facts = extract_numeric_facts(articles[0]["text"][:500], _dates)
print(f"Dates : {_dates[:3]}")
print(f"Numeric facts: {len(_facts)}")
for f in _facts[:3]: print(f"  {f.metric}: {f.raw_value} ({f.unit})")

Dates : ['2013-08-01 20:37:58']
Numeric facts: 3
  earnings: $1.92 (USD)
  earnings: $2.03 (USD)
  value: $2.04 (USD)


## 8. LLM-Based Relation Extraction — Llama 3.2 3B

Llama 3.2 3B is fed each chunk together with the already-extracted entity list.
It extracts implicit and cross-sentence relations that the BIO tagger alone misses.
Extracted relations are merged with provenance into the graph.

In [9]:
from rapidfuzz import fuzz

RELATION_TYPES = [
    "ACQUIRED", "MERGED_WITH", "INVESTED_IN", "ANNOUNCED", "REPORTED",
    "EMPLOYED_BY", "LOCATED_IN", "PARTNERED_WITH", "REGULATED_BY",
    "ISSUED", "SUBSIDIARY_OF", "COMPETES_WITH", "HAS_NUMERIC_FACT",
]

_RE_PROMPT = """You are a financial information extraction expert.
Given the news text and the named entities found in it, extract ALL meaningful relationships.

Text: {text}
Known entities: {entities}

Output ONLY valid JSON objects, one per line — no markdown, no preamble:
{{"head": "EntityA", "head_type": "ORG|PER|GPE|...", "relation": "RELATION_TYPE", "tail": "EntityB", "tail_type": "ORG|...", "confidence": 0.85, "evidence": "short quote"}}

Valid relation types: {rel_types}
Include implicit and cross-sentence relations. Output JSON lines only:"""

@dataclass
class Relation:
    head          : str
    head_type     : str
    relation_type : str
    tail          : str
    tail_type     : str
    confidence    : float
    provenance    : Optional[Provenance] = None
    time_interval : Optional[str]        = None
    numeric_facts : List[NumericFact]    = field(default_factory=list)
    source        : str                  = "llm"

def _match_entity_name(raw_name: str, entities: List[EntityMention]) -> Optional[Tuple[str, str]]:
    """Map an LLM-produced entity string back to an extracted mention."""
    candidate = _clean_entity_text(raw_name)
    if not candidate:
        return None
    best = None
    best_score = -1.0
    for e in entities:
        score = max(
            fuzz.ratio(candidate.lower(), e.text.lower()),
            fuzz.partial_ratio(candidate.lower(), e.text.lower()),
        )
        if score > best_score:
            best_score = score
            best = e
    if best is None or best_score < 86:
        return None
    return best.text, best.entity_type

def extract_relations_llm(chunk: Dict, entities: List[EntityMention], max_chars: int = 800) -> List[Relation]:
    """Use Llama 3.2 3B to extract relations from a chunk with strict grounding."""
    grounded_entities = [e for e in entities if is_valid_entity_name(e.text, e.entity_type)]
    if len(grounded_entities) < 2:
        return []

    ent_str = ", ".join(f"{e.text} ({e.entity_type})" for e in grounded_entities[:15])
    raw = ollama(
        CFG["llm_model"],
        _RE_PROMPT.format(
            text=chunk["text"][:max_chars],
            entities=ent_str,
            rel_types=", ".join(RELATION_TYPES),
        ),
        timeout=60,
    )

    rels = []
    for line in raw.strip().split("\n"):
        try:
            m = re.search(r"\{.*\}", line, re.DOTALL)
            if not m:
                continue
            obj = json.loads(m.group())
            if not all(k in obj for k in ("head", "relation", "tail")):
                continue

            mh = _match_entity_name(obj.get("head", ""), grounded_entities)
            mt = _match_entity_name(obj.get("tail", ""), grounded_entities)
            if not mh or not mt:
                continue
            head_name, head_type = mh
            tail_name, tail_type = mt
            if head_name == tail_name:
                continue

            rtype = str(obj["relation"]).upper().replace(" ", "_")
            if rtype not in RELATION_TYPES:
                continue

            confidence = float(obj.get("confidence", 0.0))
            if confidence < 0.55:
                continue

            prov = Provenance(
                chunk["chunk_id"],
                str(obj.get("evidence", ""))[:150],
                chunk["article_id"],
                chunk["pub_date"],
            )
            rels.append(
                Relation(
                    head=head_name,
                    head_type=head_type,
                    relation_type=rtype,
                    tail=tail_name,
                    tail_type=tail_type,
                    confidence=confidence,
                    provenance=prov,
                )
            )
        except Exception:
            continue
    return rels

print(f"LLM relation extractor ready. Relation taxonomy: {len(RELATION_TYPES)} types.")

LLM relation extractor ready. Relation taxonomy: 13 types.


## 9. Graph Construction — Neo4j (+ NetworkX Fallback)

An **append-only property graph** where:
- **Nodes** carry `canonical_id`, `name`, `type`, `first_seen`, `last_seen`
- **Edges** carry `relation_type`, `confidence`, `time_interval`, `provenance` (JSON), `numeric_facts`
- Parallel edges are allowed (same entity pair, different time or source)

Neo4j is used when available. All operations also maintain an in-memory `nx.MultiDiGraph`
so the rest of the pipeline works even without a running database.

In [10]:
from collections import defaultdict

G_knowledge    : nx.MultiDiGraph = nx.MultiDiGraph()
entity_registry: Dict[str, Dict] = {}
community_store: Dict[int, Dict] = {}

# Try Neo4j
NEO4J_AVAILABLE = False
neo4j_driver    = None
try:
    from neo4j import GraphDatabase
    neo4j_driver = GraphDatabase.driver(
        CFG["neo4j_uri"], auth=(CFG["neo4j_user"], CFG["neo4j_password"])
    )
    neo4j_driver.verify_connectivity()
    NEO4J_AVAILABLE = True
    print(f"Neo4j connected: {CFG['neo4j_uri']}")
except Exception as _e:
    print(f"Neo4j not available ({_e}). Using in-memory NetworkX graph.")

def canonical_id(name: str, etype: str) -> str:
    return f"{etype.lower()}_{re.sub(r'[^a-z0-9]+', '_', name.strip().lower())}"

def upsert_entity(name: str, etype: str, ts: Optional[str] = None) -> Optional[str]:
    cleaned = _clean_entity_text(name)
    if not is_valid_entity_name(cleaned, etype):
        return None

    cid = canonical_id(cleaned, etype)
    if cid not in entity_registry:
        entity_registry[cid] = {
            "canonical_id": cid,
            "name": cleaned,
            "type": etype,
            "first_seen": ts,
            "last_seen": ts,
        }
        G_knowledge.add_node(cid, **entity_registry[cid])
    elif ts and (not entity_registry[cid]["last_seen"] or ts > entity_registry[cid]["last_seen"]):
        entity_registry[cid]["last_seen"] = ts
        if G_knowledge.has_node(cid):
            G_knowledge.nodes[cid]["last_seen"] = ts
    return cid

def add_relation(rel: Relation) -> None:
    hid = upsert_entity(rel.head, rel.head_type, rel.time_interval)
    tid = upsert_entity(rel.tail, rel.tail_type, rel.time_interval)
    if not hid or not tid or hid == tid:
        return

    props = {
        "relation_type": rel.relation_type,
        "confidence": rel.confidence,
        "time_interval": rel.time_interval or "",
        "source": rel.source,
        "provenance": json.dumps(rel.provenance.to_dict()) if rel.provenance else "{}",
        "numeric_facts": json.dumps([nf.to_dict() for nf in rel.numeric_facts]),
    }
    G_knowledge.add_edge(hid, tid, **props)

    if NEO4J_AVAILABLE:
        try:
            with neo4j_driver.session() as sess:
                sess.run(
                    """
                    MERGE (h:Entity {canonical_id:$hid}) SET h.name=$hn, h.type=$ht
                    MERGE (t:Entity {canonical_id:$tid}) SET t.name=$tn, t.type=$tt
                    CREATE (h)-[r:RELATION {relation_type:$rt, confidence:$cf,
                        time_interval:$ti, source:$src, provenance:$pv}]->(t)
                    """,
                    hid=hid, hn=entity_registry[hid]["name"], ht=rel.head_type,
                    tid=tid, tn=entity_registry[tid]["name"], tt=rel.tail_type,
                    rt=rel.relation_type, cf=rel.confidence,
                    ti=props["time_interval"], src=rel.source, pv=props["provenance"],
                )
        except Exception as neo_e:
            logger.warning("Neo4j write: %s", neo_e)

def add_numeric_edge(ent_name: str, etype: str, fact: NumericFact, prov: Provenance):
    # Keep numeric self-loops only on stronger anchor entity types to reduce noise.
    if etype not in {"ORG", "PER", "GPE", "PRODUCT"}:
        return
    if not fact or not fact.metric or fact.metric == "value":
        return

    cid = upsert_entity(ent_name, etype, fact.time_interval)
    if not cid:
        return

    G_knowledge.add_edge(
        cid, cid,
        relation_type="HAS_NUMERIC_FACT",
        metric=fact.metric, raw_value=fact.raw_value, unit=fact.unit,
        time_interval=fact.time_interval or "",
        provenance=json.dumps(prov.to_dict()),
        confidence=1.0, source="rule",
        numeric_facts=json.dumps([fact.to_dict()]),
    )

print("Graph builder ready.")

Neo4j not available (Couldn't connect to localhost:7687 (resolved to ('127.0.0.1:7687',)):
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)). Using in-memory NetworkX graph.
Graph builder ready.


## 10. Knowledge Graph Construction — Full Pipeline per Chunk

For each chunk:
1. **FinBERT + BIO** → entity mentions + embeddings
2. **SSHG** → entity-level graph from dependency + co-occurrence + cosine similarity
3. **GCN** → refine entity embeddings via neighbourhood aggregation
4. **Temporal / Numeric** → extract dates and numeric facts
5. **LLM RE** (every 10th chunk) → implicit relations via Llama 3.2 3B
6. **Graph write** → upsert entities + edges into `G_knowledge` (and Neo4j if available)

> Set `CFG["process_n_chunks"] = len(article_chunks)` to process the full 5 000-article corpus.

In [11]:
PROCESS_N = min(CFG["process_n_chunks"], len(article_chunks))
print(f"Building KG from {PROCESS_N:,} / {len(article_chunks):,} chunks ...")
print("(Increase CFG['process_n_chunks'] for a larger graph)\n")

_errors = 0
for _idx, chunk in enumerate(tqdm(article_chunks[:PROCESS_N], desc="KG build")):
    try:
        # 1. FinBERT NER
        ents, _ = extract_entities_bio(chunk["text"])
        if not ents:
            continue

        # 2. SSHG
        sshg = build_sshg(chunk["text"], ents)

        # 3. GCN — refine embeddings
        if sshg.number_of_nodes() > 1:
            refined = apply_gcn(ents, sshg)
            for _j, ent in enumerate(ents):
                if _j < len(refined):
                    ent.embedding = refined[_j]

        # 4. Temporal + numeric
        dates    = extract_dates(chunk["text"], chunk["pub_date"])
        num_facts= extract_numeric_facts(chunk["text"], dates)
        ts       = dates[0] if dates else chunk["pub_date"]

        prov = Provenance(chunk["chunk_id"], chunk["text"][:150],
                          chunk["article_id"], chunk["pub_date"])

        # 5. LLM relation extraction (every 10th chunk to manage latency)
        llm_rels = []
        if _idx % 10 == 0:
            llm_rels = extract_relations_llm(chunk, ents)

        # 6. Write to graph
        for ent in ents:
            upsert_entity(ent.text, ent.entity_type, ts)

        for rel in llm_rels:
            rel.time_interval = ts
            rel.numeric_facts = num_facts[:2]
            add_relation(rel)

        # Numeric self-loop edges for top entities
        for ent in ents[:3]:
            for nf in num_facts[:2]:
                add_numeric_edge(ent.text, ent.entity_type, nf, prov)

    except Exception as _e:
        _errors += 1
        logger.warning("Chunk %s error: %s", chunk["chunk_id"], _e)

print(f"\nKnowledge Graph built:")
print(f"  Entities (nodes)  : {G_knowledge.number_of_nodes():,}")
print(f"  Relations (edges) : {G_knowledge.number_of_edges():,}")
print(f"  Errors skipped    : {_errors}")

Building KG from 500 / 33,131 chunks ...
(Increase CFG['process_n_chunks'] for a larger graph)



KG build: 100%|██████████| 500/500 [19:28<00:00,  2.34s/it]


Knowledge Graph built:
  Entities (nodes)  : 1,830
  Relations (edges) : 281
  Errors skipped    : 0


## 11. Hierarchy & Communities — Leiden Algorithm

The Leiden algorithm partitions the entity graph into communities, which become
the global retrieval index. Each community receives an LLM-generated summary
grounded exclusively in intra-community relations.

In [12]:
try:
    import igraph as ig
    import leidenalg
    _LEIDEN_OK = True
except ImportError:
    _LEIDEN_OK = False
    print("leidenalg/igraph not found — install: pip install leidenalg python-igraph")

_SUMMARY_PROMPT = """You are a financial knowledge analyst.
Summarise this cluster of related financial entities and their connections in 2-3 sentences.
Focus on the financial theme or event that links these entities.

Entities: {entities}
Relations: {relations}

Concise summary (2-3 sentences, facts only):"""

def run_leiden(G: nx.MultiDiGraph) -> Dict[str, int]:
    if not _LEIDEN_OK or G.number_of_nodes() == 0:
        return {}
    nodes     = list(G.nodes())
    n2i       = {n: i for i, n in enumerate(nodes)}
    undirected= G.to_undirected()
    edges_ig  = [(n2i[u], n2i[v]) for u, v in undirected.edges()
                 if u in n2i and v in n2i]
    g_ig      = ig.Graph(n=len(nodes), edges=edges_ig, directed=False)
    g_ig.simplify()
    part      = leidenalg.find_partition(g_ig, leidenalg.ModularityVertexPartition)
    return {nodes[i]: part.membership[i] for i in range(len(nodes))}

def build_community_summaries(G: nx.MultiDiGraph, membership: Dict[str, int],
                               cap: int = CFG["max_communities"]) -> Dict[int, Dict]:
    by_comm: Dict[int, List[str]] = defaultdict(list)
    for node, cid in membership.items():
        by_comm[cid].append(G.nodes[node].get("name", node))

    summaries: Dict[int, Dict] = {}
    for cid in tqdm(sorted(by_comm)[:cap], desc="Community summaries"):
        ents = by_comm[cid]
        comm_nodes = {n for n, c in membership.items() if c == cid}
        rels_txt = []
        for u, v, d in G.edges(data=True):
            if u in comm_nodes and v in comm_nodes:
                un = G.nodes[u].get("name", u)
                vn = G.nodes[v].get("name", v)
                rels_txt.append(f"{un} -{d.get('relation_type','REL')}-> {vn}")
        summary = ollama(
            CFG["llm_model"],
            _SUMMARY_PROMPT.format(entities=", ".join(ents[:10]),
                                   relations="; ".join(rels_txt[:10]) or "none"),
            timeout=45,
        )
        summaries[cid] = {"community_id": cid, "entities": ents,
                          "n_entities": len(ents), "summary": summary}
    return summaries

print("Running Leiden community detection ...")
if G_knowledge.number_of_nodes() > 0:
    membership = run_leiden(G_knowledge)
    n_comms = len(set(membership.values()))
    print(f"Communities found: {n_comms}")
    for node, cid in membership.items():
        if G_knowledge.has_node(node):
            G_knowledge.nodes[node]["community_id"] = cid
    print(f"Generating summaries (cap={CFG['max_communities']}) ...")
    community_store = build_community_summaries(G_knowledge, membership)
    print(f"Summaries generated: {len(community_store)}")
    if community_store:
        _sc = next(iter(community_store.values()))
        print(f"\nSample community ({_sc['n_entities']} entities):")
        print(f"  {', '.join(_sc['entities'][:5])}")
        print(f"  {_sc['summary'][:200]}")
else:
    membership = {}
    print("Graph empty — skipping Leiden.")

Running Leiden community detection ...
Communities found: 1707
Generating summaries (cap=30) ...


Community summaries: 100%|██████████| 30/30 [01:42<00:00,  3.41s/it]

Summaries generated: 30

Sample community (15 entities):
  U.S, the Middle East, India, China, North Africa
  The financial entities connected by this cluster are primarily related to grain trade and regulation. The United States is linked to both the Russian Grain Union and the US Department of Agriculture (


---
# Part 2 — Knowledge Retrieval & QA (Inference)

Given a user question, the pipeline:
1. Classifies the query as `local` / `global` / `mixed`
2. Routes to the appropriate retriever
3. Generates an answer with **Mistral-NeMo** using only the retrieved context
4. Evaluates with the **RAG Triad** (Faithfulness, Answer Relevance, Context Precision)

## 12. Query Type Determination

In [13]:
_QT_PROMPT = """Classify this financial question into exactly one type:
- local  : asks about a specific entity (company, person, place)
- global : asks about broad themes, trends, or sector-wide patterns
- mixed  : compares entities or combines specific + thematic reasoning

Question: {question}

Output ONE word only (local / global / mixed):"""

def classify_query(question: str) -> str:
    raw = ollama(CFG["llm_model"], _QT_PROMPT.format(question=question)).lower()
    for qt in ("local", "global", "mixed"):
        if qt in raw:
            return qt
    return "local"

# Smoke test
for _q, _expect in [
    ("What did Goldman Sachs acquire in 2022?", "local"),
    ("What were major acquisition trends in 2022?", "global"),
]:
    _got = classify_query(_q)
    print(f"  '{_q[:55]}...' -> {_got}  (expected {_expect})")

  'What did Goldman Sachs acquire in 2022?...' -> local  (expected local)
  'What were major acquisition trends in 2022?...' -> mixed  (expected global)


## 13. Query Router — Local / Global / Hybrid Retrieval

| Route | Mechanism |
|-------|-----------|
| **Local** | Extract entity names from query → map to canonical IDs → BFS 2-hop subgraph → serialise as text |
| **Global** | Embed query with `all-MiniLM-L6-v2` → cosine similarity vs community summary embeddings → top-k |
| **Hybrid** | Concatenate local subgraph + global community summaries |

In [14]:
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {CFG['embed_model']} ...")
embedder = SentenceTransformer(CFG["embed_model"], device="cpu")
print("Embedding model ready.")

# Pre-compute community summary embeddings
_comm_list = list(community_store.values())
if _comm_list:
    _comm_texts = [c["summary"] for c in _comm_list]
    _comm_embs  = embedder.encode(_comm_texts, normalize_embeddings=True,
                                   show_progress_bar=False).astype(np.float32)
    print(f"Community embeddings: {_comm_embs.shape}")
else:
    _comm_embs = np.zeros((0, 384), dtype=np.float32)
    print("No community summaries — global retrieval will return empty context.")

# Common words that must never be treated as entity matches
_STOP_ENTS = {
    "the","a","an","of","in","on","at","by","for","to","is","are","was","were",
    "be","been","being","have","has","had","do","does","did","will","would",
    "could","should","may","might","shall","what","which","who","whom","whose",
    "when","where","why","how","this","that","these","those","it","its","i",
    "he","she","we","they","you","me","him","her","us","them","my","your",
    "his","our","their","year","quarter","percent","period","time","day",
    "month","week","date","company","group","inc","corp","ltd","plc","co",
    "said","says","new","first","last","also","just","more","most","some",
    "planned","outstanding","friend","list","s","the",
}

def _valid_entity(name: str) -> bool:
    """Return True only for plausible proper-noun entity names."""
    n = name.strip()
    if len(n) < 4:                          return False
    if n.lower() in _STOP_ENTS:             return False
    if not re.search(r"[A-Za-z]", n):       return False
    if not re.search(r"[A-Z]", n):          return False
    words = n.lower().split()
    if len(words) > 1 and all(w in _STOP_ENTS for w in words): return False
    return True

# ── Local retrieval
def local_retrieval(question: str, hop: int = CFG["hop"]) -> str:
    q_lower = question.lower()

    # Pass 1: valid entity name is substring of question
    matched = [
        cid for cid, attrs in entity_registry.items()
        if _valid_entity(attrs["name"])
        and attrs["name"].lower() in q_lower
    ]

    # Pass 2: capitalised words from question overlap valid entity names
    if not matched:
        q_words = set(re.findall(r'\b[A-Z][A-Za-z]{2,}\b', question))
        matched = [
            cid for cid, attrs in entity_registry.items()
            if _valid_entity(attrs["name"])
            and any(w.lower() in attrs["name"].lower() or
                    attrs["name"].lower() in w.lower()
                    for w in q_words)
        ]

    matched = matched[:5]
    if not matched:
        return ""

    # BFS k-hop subgraph
    sub = set(matched)
    frontier = set(matched)
    for _ in range(hop):
        nxt = set()
        for node in frontier:
            if G_knowledge.has_node(node):
                nxt.update(G_knowledge.predecessors(node))
                nxt.update(G_knowledge.successors(node))
        sub.update(nxt)
        frontier = nxt - sub

    lines = []
    for u, v, d in G_knowledge.edges(data=True):
        if u not in sub or v not in sub:
            continue
        un  = entity_registry.get(u, {}).get("name", u)
        vn  = entity_registry.get(v, {}).get("name", v)
        rt  = d.get("relation_type", "REL")
        ts  = d.get("time_interval", "")
        cf  = d.get("confidence", 0.0)
        try:
            sent = json.loads(d.get("provenance","{}")).get("sentence","")[:80]
        except Exception:
            sent = ""
        line = f"({un}) -[{rt}]-> ({vn})"
        if ts:   line += f" [{ts}]"
        if cf:   line += f" conf={cf:.2f}"
        if sent: line += f' | "{sent}"'
        lines.append(line)
    return "\n".join(lines[:40]) if lines else (
        "Entities found: " + ", ".join(entity_registry.get(n,{}).get("name",n) for n in matched))

# ── Global retrieval
def global_retrieval(question: str, k: int = CFG["top_k_global"]) -> str:
    if _comm_embs.shape[0] == 0:
        return ""
    qv   = embedder.encode([question], normalize_embeddings=True)[0].astype(np.float32)
    sims = _comm_embs @ qv
    idxs = np.argsort(-sims)[:k]
    parts = []
    for i in idxs:
        if i < len(_comm_list):
            c = _comm_list[int(i)]
            parts.append(f"[Community {c['community_id']} | {c['n_entities']} entities]\n{c['summary']}")
    return "\n\n".join(parts)

# ── Hybrid
def hybrid_retrieval(question: str) -> str:
    parts = []
    lc = local_retrieval(question)
    gc = global_retrieval(question)
    if lc: parts.append(f"=== Local Subgraph ===\n{lc}")
    if gc: parts.append(f"=== Global Communities ===\n{gc}")
    return "\n\n".join(parts)

def route_and_retrieve(question: str) -> Tuple[str, str]:
    """Returns (context_text, query_type)."""
    qt = classify_query(question)
    ctx = (local_retrieval(question)  if qt == "local"
           else global_retrieval(question) if qt == "global"
           else hybrid_retrieval(question))
    return ctx, qt

print("Retrieval functions ready.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


2026-04-15 00:40:15,366 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 00:40:15,591 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-15 00:40:15,892 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 00:40:16,048 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-15 00:40:16,050 INFO Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-04-15 00:40:16,205 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all

Embedding model ready.
Community embeddings: (30, 384)
Retrieval functions ready.


## 14. Answer Generation — Mistral-NeMo 12B

The model is instructed to answer **only from the provided context** and to say
"I don't know" when the context does not support an answer.
For conversational follow-ups, the question is first rewritten as standalone.

In [15]:
_REWRITE_PROMPT = """Given the conversation history and a follow-up question,
rewrite the follow-up as a fully standalone question.

History:
{history}

Follow-up: {question}

Standalone question:"""

_ANSWER_PROMPT = """You are a precise financial analyst.
Answer the question using ONLY the provided context.
If the context does not support an answer, say "I don't know based on the available information."
Do not add any information not present in the context.

Context:
{context}

Question: {question}

Answer:"""

def generate_answer(question: str, context: str,
                    history: Optional[List[Dict]] = None) -> str:
    """Generate answer with Mistral-NeMo. Rewrites question if history is provided."""
    if history:
        hist_str = "\n".join(f"Q: {h['question']}\nA: {h['answer']}" for h in history[-3:])
        question = ollama(CFG["answer_model"],
                          _REWRITE_PROMPT.format(history=hist_str, question=question),
                          timeout=60)
    ctx = context.strip() or "No relevant information found in the knowledge graph."
    return ollama(CFG["answer_model"],
                  _ANSWER_PROMPT.format(context=ctx[:3000], question=question),
                  timeout=90)

print("Answer generator ready (model: {})".format(CFG["answer_model"]))

Answer generator ready (model: mistral-nemo:latest)


## 15. Load QA Set & Run Full Pipeline

Load the shared `qa_set.json` (generated in Notebook 1) and run all 20 questions
through the complete GraphRAG pipeline.

In [16]:
if not os.path.exists(CFG["qa_set_path"]):
    raise FileNotFoundError(
        f"{CFG['qa_set_path']} not found. Run Baseline 1 (Pure LLM) notebook first.")

with open(CFG["qa_set_path"]) as _f:
    qa_set = json.load(_f)

print(f"QA set loaded: {len(qa_set)} questions")
for qa in qa_set[:5]:
    print(f"  [{qa['qa_id']:2d}] ({qa['topic']:8s}) {qa['question'][:65]}")

results: List[Dict] = []
_t0 = time.time()

print(f"\nRunning GraphRAG on {len(qa_set)} questions ...")
for qa in tqdm(qa_set, desc="GraphRAG QA"):
    _ts = time.perf_counter()
    ctx, qtype = route_and_retrieve(qa["question"])
    answer     = generate_answer(qa["question"], ctx)
    latency    = (time.perf_counter() - _ts) * 1000

    results.append({
        "qa_id"            : qa["qa_id"],
        "article_id"       : qa["article_id"],
        "topic"            : qa["topic"],
        "question"         : qa["question"],
        "reference_answer" : qa["reference_answer"],
        "predicted_answer" : answer,
        "context_used"     : ctx[:600],
        "query_type"       : qtype,
        "latency_ms"       : round(latency, 1),
        "method"           : "graphrag",
    })

_elapsed = time.time() - _t0
print(f"\nDone: {len(results)} answers in {_elapsed:.0f}s ({_elapsed/len(results):.1f}s each)")

# Preview first result
_r0 = results[0]
print(f"\nQ : {_r0['question']}")
print(f"A : {_r0['predicted_answer'][:200]}")
print(f"Ctx: {_r0['context_used'][:150]}")

QA set loaded: 20 questions
  [ 0] (earnings) What was Marriott's new projected full-year earnings per share ra
  [ 1] (earnings) What is the planned date for the listing of Health, the new entit
  [ 2] (other   ) What is the current outstanding amount owed on the Friendly Seas 
  [ 3] (policy  ) What percentage of Americans did not have health insurance in 201
  [ 4] (policy  ) What was the date when the Swiss National Bank introduced the cap

Running GraphRAG on 20 questions ...


GraphRAG QA: 100%|██████████| 20/20 [04:30<00:00, 13.52s/it]


Done: 20 answers in 270s (13.5s each)

Q : What was Marriott's new projected full-year earnings per share range?
A : I don't know based on the available information.
Ctx: Entities found: Marriott, Marriott


## 16. RAG Triad Evaluation — LLM-as-Judge (Llama 3.2 3B)

| Metric | Definition |
|--------|------------|
| **Faithfulness** | Every claim in the answer is supported by the retrieved context |
| **Answer Relevance** | Answer directly addresses the question |
| **Context Precision** | Fraction of retrieved context useful for answering |

In [17]:
_FAITH = ("Rate faithfulness of the answer to the context (0.0-1.0).\n"
          "Faithfulness = every claim is supported by the context.\n\n"
          "Context: {context}\nAnswer: {answer}\n\n"
          "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}")
_RELEV = ("Rate relevance of the answer to the question (0.0-1.0).\n\n"
          "Question: {query}\nAnswer: {answer}\n\n"
          "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}")
_PREC  = ("Rate context precision (0.0-1.0).\n"
          "Precision = fraction of context useful for answering the question.\n\n"
          "Question: {query}\nContext: {context}\n\n"
          "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}")

def parse_score(raw: str) -> float:
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return float(json.loads(m.group())["score"]) if m else 0.0
    except Exception:
        return 0.0

print(f"Evaluating {len(results)} answers with {CFG['llm_model']} judge ...")
for r in tqdm(results, desc="Triad eval"):
    ctx_p = r["context_used"][:600]
    faith = parse_score(ollama(CFG["llm_model"],
                               _FAITH.format(context=ctx_p, answer=r["predicted_answer"])))
    relev = parse_score(ollama(CFG["llm_model"],
                               _RELEV.format(query=r["question"], answer=r["predicted_answer"])))
    prec  = parse_score(ollama(CFG["llm_model"],
                               _PREC.format(query=r["question"], context=ctx_p)))
    r["faithfulness"]      = round(faith, 3)
    r["answer_relevance"]  = round(relev, 3)
    r["context_precision"] = round(prec,  3)
    r["triad_avg"]         = round((faith + relev + prec) / 3, 3)

print()
print("=" * 60)
print("GRAPHRAG — BENCHMARK RESULTS")
print("=" * 60)
print(f"{'Metric':28s}  {'Mean':>7}  {'Std':>7}")
print("-" * 45)
for _key in ["faithfulness","answer_relevance","context_precision","triad_avg"]:
    _vals = [r[_key] for r in results]
    print(f"{_key:28s}  {np.mean(_vals):7.4f}  {np.std(_vals):7.4f}")
print(f"{'latency_ms (avg)':28s}  {np.mean([r['latency_ms'] for r in results]):7.1f}")

Evaluating 20 answers with llama3.2:3b judge ...


Triad eval: 100%|██████████| 20/20 [01:57<00:00,  5.88s/it]


GRAPHRAG — BENCHMARK RESULTS
Metric                           Mean      Std
---------------------------------------------
faithfulness                   0.1500   0.3571
answer_relevance               0.0500   0.2179
context_precision              0.3300   0.3333
triad_avg                      0.1767   0.1559
latency_ms (avg)              13521.0


## 17. Save Results & Three-Way Comparison

In [18]:
summary = {
    "method"           : "GraphRAG (FinBERT+GCN+SSHG+Leiden+Mistral-Nemo)",
    "n_questions"      : len(results),
    "kg_nodes"         : G_knowledge.number_of_nodes(),
    "kg_edges"         : G_knowledge.number_of_edges(),
    "n_communities"    : len(community_store),
    "mean_faithfulness": float(np.mean([r["faithfulness"]      for r in results])),
    "mean_relevance"   : float(np.mean([r["answer_relevance"]  for r in results])),
    "mean_ctx_precision": float(np.mean([r["context_precision"] for r in results])),
    "mean_triad_avg"   : float(np.mean([r["triad_avg"]         for r in results])),
    "mean_latency_ms"  : float(np.mean([r["latency_ms"]        for r in results])),
    "per_question"     : results,
}
with open(CFG["results_path"], "w") as _f:
    json.dump(summary, _f, indent=2)
print(f"Saved: {CFG['results_path']} ({os.path.getsize(CFG['results_path'])//1024} KB)")

# ── Three-way comparison ──────────────────────────────────────────────────────
def _load_baseline(path):
    if not os.path.exists(path): return {}
    with open(path) as f: d = json.load(f)
    return {
        "faithfulness"     : d.get("mean_faithfulness",   d.get("mean_faith", 0)),
        "answer_relevance" : d.get("mean_relevance",      d.get("mean_answer_relevance", 0)),
        "context_precision": d.get("mean_ctx_precision",  d.get("mean_context_precision", 0)),
        "triad_avg"        : d.get("mean_triad_avg", 0),
    }

_llm = _load_baseline("baseline_llm_results.json")
_rag = _load_baseline("baseline_rag_results.json")
_gr  = {k: float(np.mean([r[k] for r in results]))
        for k in ["faithfulness","answer_relevance","context_precision","triad_avg"]}

print()
print("=" * 78)
print("FULL BENCHMARK: Pure LLM  |  Vanilla RAG  |  GraphRAG")
print("=" * 78)
print(f"{'Metric':26s}  {'Pure LLM':>10}  {'Vanilla RAG':>12}  {'GraphRAG':>9}  {'Δ vs RAG':>9}")
print("-" * 72)
for _k in ["faithfulness","answer_relevance","context_precision","triad_avg"]:
    _vl = _llm.get(_k, float("nan"))
    _vr = _rag.get(_k, float("nan"))
    _vg = _gr[_k]
    _d  = _vg - _vr
    print(f"{_k:26s}  {_vl:10.4f}  {_vr:12.4f}  {_vg:9.4f}  {'+' if _d>=0 else ''}{_d:7.4f}")
print()
print("GraphRAG graph stats:")
print(f"  Entities  : {G_knowledge.number_of_nodes():,}")
print(f"  Relations : {G_knowledge.number_of_edges():,}")
print(f"  Communities: {len(community_store)}")

Saved: graphrag_results.json (14 KB)

FULL BENCHMARK: Pure LLM  |  Vanilla RAG  |  GraphRAG
Metric                        Pure LLM   Vanilla RAG   GraphRAG   Δ vs RAG
------------------------------------------------------------------------
faithfulness                       nan           nan     0.1500      nan
answer_relevance                   nan           nan     0.0500      nan
context_precision                  nan           nan     0.3300      nan
triad_avg                          nan           nan     0.1767      nan

GraphRAG graph stats:
  Entities  : 1,830
  Relations : 281
  Communities: 30


## Summary

| Component | Choice | Role |
|-----------|--------|------|
| Encoder | `ProsusAI/finbert` | Domain-adapted token embeddings for financial text |
| NER | BIO head on FinBERT | Entity span detection with financial entity types |
| Graph structure | SSHG (syntactic + co-occurrence + cosine) | Rich entity-level connectivity |
| Propagation | 2-layer GCN | Neighbourhood-informed entity representations |
| Relation extraction | Llama 3.2 3B (Ollama) | Implicit / cross-sentence relations |
| Graph store | Neo4j (+ NetworkX fallback) | Append-only property graph with provenance |
| Community detection | Leiden algorithm | Hierarchical global retrieval index |
| Community summaries | Llama 3.2 3B | Concise cluster descriptions for global retrieval |
| Query routing | local / global / hybrid | Adapts retrieval strategy to question type |
| Answer model | Mistral-NeMo (Ollama) | Context-grounded generation |
| Evaluation | RAG Triad via Llama 3.2 3B | Faithfulness · Relevance · Precision |

**Output files:**
- `graphrag_results.json` — per-question answers + triad scores
- Compare with `baseline_llm_results.json` and `baseline_rag_results.json`